In [ ]:
def create_all_features(data, cycle=CYCLE, drop_incomplete=True):
    df_feat = data.copy()

    # calendar features: hour of day, and hour as a cycle (sin/cos)
    df_feat["hour"] = df_feat.index.hour
    df_feat["sin_hour"] = np.sin(2 * np.pi * df_feat.index.hour / 24)
    df_feat["cos_hour"] = np.cos(2 * np.pi * df_feat.index.hour / 24)

    # trend features: position in time (counted from the start of the series) and its square
    df_feat["trend"] = ((df_feat.index - ml_df.index[0]) / pd.Timedelta(minutes=1)).to_numpy()
    df_feat["trend_squared"] = df_feat["trend"] ** 2

    # lag features: recent minutes plus one full fill/drain cycle ago
    for lag in sorted({1, 2, 3, 6, 12, 24, cycle}):
        df_feat[f"lag_{lag}"] = df_feat[target].shift(lag)

    # rolling mean and std of the previous window (shift(1) excludes the current value)
    for window in [3, 6, 12]:
        df_feat[f"rolling_mean_{window}"] = df_feat[target].shift(1).rolling(window).mean()
        df_feat[f"rolling_std_{window}"] = df_feat[target].shift(1).rolling(window).std()

    # ratio: last value relative to one cycle ago
    df_feat[f"ratio_lag1_lag{cycle}"] = df_feat["lag_1"] / df_feat[f"lag_{cycle}"].replace(0, np.nan)


    feature_cols = df_feat.columns.drop(target)
    df_feat[feature_cols] = df_feat[feature_cols].replace([np.inf, -np.inf], np.nan)
    return df_feat.dropna(subset=feature_cols) if drop_incomplete else df_feat

# Each feature at t uses only values before t. Carry training history into the
# test boundary; dropping the first CYCLE test rows would discard valid forecasts.
features = create_all_features(ml_df)
train_feat = features.loc[features.index < TEST_START]
test_feat = features.loc[features.index >= TEST_START]
X_train_full, y_train = train_feat.drop(columns=target), train_feat[target]
X_test_full, y_test = test_feat.drop(columns=target), test_feat[target]
if not y_test.index.equals(test.index):
    raise ValueError('Feature construction unexpectedly removed test minutes.')
split = int((X_train_full.index < VAL_START).sum())
X_fit_full, X_val_full = X_train_full.iloc[:split], X_train_full.iloc[split:]
y_train_final, y_val_final = y_train.iloc[:split], y_train.iloc[split:]
y_test_final = y_test
print('Features:', X_train_full.shape[1], '| fit / validation / test:',
      len(X_fit_full), len(X_val_full), len(X_test_full))


In [ ]:
# score each feature with a univariate F-test against the target, keep the best 8
filter_selector = SelectKBest(score_func=f_regression, k=min(8, X_train_full.shape[1]))
filter_selector.fit(X_fit_full, y_train_final)
filter_selected = X_train_full.columns[filter_selector.get_support()].tolist()
print(f"Filter selected: {filter_selected}")
#The filter method scores every feature independently by its relationship with 
#the target and keeps the 8 highest-scoring features.


In [ ]:
# wrapper: fit a random forest, remove the least important feature, repeat until 8 remain
estimator = RandomForestRegressor(n_estimators=100, random_state=42)
wrapper_selector = RFE(estimator, n_features_to_select=min(8, X_train_full.shape[1]))
wrapper_selector.fit(X_fit_full, y_train_final)
wrapper_selected = X_train_full.columns[wrapper_selector.support_].tolist()
print(f"Wrapper selected: {wrapper_selected}")


In [ ]:
selection_methods = {'Filter': filter_selected, 'Wrapper': wrapper_selected,
                     'All Features': X_train_full.columns.tolist()}
method_results = {}
for method_name, feats in selection_methods.items():
    selection_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    selection_model.fit(X_fit_full[feats], y_train_final)
    pred = selection_model.predict(X_val_full[feats])
    method_results[method_name] = {
        'rmse': root_mean_squared_error(y_val_final, pred),
        'mae': mean_absolute_error(y_val_final, pred),
        'r2': r2_score(y_val_final, pred), 'features': feats, 'n_features': len(feats)}
print('Feature-subset comparison on validation only:')
display(pd.DataFrame(method_results).T[['rmse','mae','r2','n_features']].astype(float).round(4))


In [ ]:
# Select the feature selection method with the lowest RMSE
best_method = min(method_results, key=lambda x: method_results[x]['rmse'])
selected_features = method_results[best_method]['features']

print(f"\nBEST METHOD: {best_method}")
print(f"   RMSE: {method_results[best_method]['rmse']:.2f}")
print(f"   Features ({len(selected_features)}): {selected_features}")

# Keep only the selected features for model training
X_train = X_train_full[selected_features]
X_test = X_test_full[selected_features]

#The feature set achieving the lowest Root Mean Squared Error (RMSE) is selected and applied to the training 
#and testing data used by the machine learning models.


In [ ]:
X_train_final = X_fit_full[selected_features]
X_val_final = X_val_full[selected_features]
X_test_final = X_test_full[selected_features]


# Univariate

In [ ]:
processed_data = processed_clean.copy() # LIT101

In [ ]:
#Univariate Statistical Forecasting ON ARIMA

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from scipy.fft import fft, fftfreq
from sklearn.preprocessing import StandardScaler

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 4)

uni_target = "LIT101"
uni_plot_hours = 12

uni_labels = (
    processed_clean["Normal/Attack"]
    .astype("string")
    .str.strip()
)

uni_attack_mask = uni_labels.eq("Attack")

if not uni_attack_mask.any():
    raise ValueError("No attack boundary found in processed_clean.")

uni_first_attack = processed_clean.index[uni_attack_mask].min()

uni_normal_data = processed_clean.loc[
    processed_clean.index < uni_first_attack
].copy()

uni_data = pd.to_numeric(
    uni_normal_data[uni_target],
    errors="coerce"
).sort_index()

if not uni_data.index.is_unique:
    raise ValueError(
        "Duplicate timestamps remain. Resolve them before ARIMA fitting."
    )

uni_data = uni_data.asfreq("1min")

print("Selected target:", uni_target)
print("Missing values:", uni_data.isna().sum())
print("Number of observations:", len(uni_data))
print("Start time:", uni_data.index.min())
print("End time:", uni_data.index.max())

display(uni_data.describe())

if uni_data.isna().any():
    raise ValueError(
        "Missing LIT101 readings detected. Investigate the gaps "
        "before fitting ARIMA; this section does not invent readings."
    )

uni_train_full_size = int(len(uni_data) * 0.8)

uni_train_full = uni_data.iloc[:uni_train_full_size].copy()
uni_test = uni_data.iloc[uni_train_full_size:].copy()

uni_inner_split = int(len(uni_train_full) * 0.8)

uni_train = uni_train_full.iloc[:uni_inner_split].copy()
uni_validation = uni_train_full.iloc[uni_inner_split:].copy()

print("\nTraining observations:", len(uni_train))
print("Validation observations:", len(uni_validation))
print("Full development observations:", len(uni_train_full))
print("Test observations:", len(uni_test))

print("\nTraining:", uni_train.index[0], "to", uni_train.index[-1])
print("Validation:", uni_validation.index[0], "to", uni_validation.index[-1])
print("Test:", uni_test.index[0], "to", uni_test.index[-1])


uni_fft_values = fft(
    (uni_train - uni_train.mean()).to_numpy()
)

uni_frequencies = fftfreq(
    len(uni_train),
    d=1
)

uni_band = (
    (uni_frequencies > 1 / 360)
    & (uni_frequencies < 1 / 5)
)

if not uni_band.any():
    raise ValueError("No suitable FFT frequencies were found.")

uni_period = int(round(
    1 / uni_frequencies[uni_band][
        np.argmax(np.abs(uni_fft_values[uni_band]))
    ]
))

print("\nDominant training cycle:", uni_period, "minutes")


In [ ]:
#Exploration

uni_plot_start = (
    uni_train.index[-1]
    - pd.Timedelta(hours=uni_plot_hours)
)

uni_plot_end = uni_train.index[-1]

uni_train_zoom = uni_train.loc[
    uni_plot_start:uni_plot_end
]

uni_rolling_mean = uni_train.rolling(window=60).mean()
uni_rolling_var = uni_train.rolling(window=60).var()

fig, axes = plt.subplots(
    3, 1,
    figsize=(14, 9),
    sharex=True
)

axes[0].plot(
    uni_train_zoom.index,
    uni_train_zoom,
    linewidth=0.8
)
axes[0].set_title(
    f"{uni_target} - Observed Series (Last 12 Hours)",
    loc="left"
)
axes[0].set_ylabel("Water Level")
axes[0].grid(True, linestyle="--", alpha=0.3)

axes[1].plot(
    uni_rolling_mean.loc[uni_plot_start:uni_plot_end].index,
    uni_rolling_mean.loc[uni_plot_start:uni_plot_end],
    linewidth=0.9
)
axes[1].set_title("60-minute Rolling Mean", loc="left")
axes[1].set_ylabel("Rolling Mean")
axes[1].grid(True, linestyle="--", alpha=0.3)

axes[2].plot(
    uni_rolling_var.loc[uni_plot_start:uni_plot_end].index,
    uni_rolling_var.loc[uni_plot_start:uni_plot_end],
    linewidth=0.9
)
axes[2].set_title("60-minute Rolling Variance", loc="left")
axes[2].set_ylabel("Rolling Variance")
axes[2].set_xlabel("Timestamp")
axes[2].grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

uni_zscore = (
    (uni_train - uni_train.mean())
    / uni_train.std()
)

print(
    "Training observations with |z-score| > 3:",
    int((np.abs(uni_zscore) > 3).sum())
)


In [ ]:
#Decomposition

if uni_period >= 2 and 2 * uni_period < len(uni_train):

    uni_decomposition = seasonal_decompose(
        uni_train,
        model="additive",
        period=uni_period,
        extrapolate_trend="freq"
    )

    uni_components = pd.DataFrame({
        "Observed": uni_decomposition.observed,
        "Trend": uni_decomposition.trend,
        "Seasonal": uni_decomposition.seasonal,
        "Residual": uni_decomposition.resid
    })

    uni_zoom_start = (
        uni_train.index[-1]
        - pd.Timedelta(hours=uni_plot_hours)
    )

    uni_zoom = uni_components.loc[
        uni_zoom_start:uni_train.index[-1]
    ]

    uni_plot_settings = [
        ("Observed", "Observed LIT101", "Water Level"),
        ("Trend", "Trend Component", "Trend"),
        ("Seasonal", "Seasonal Component", "Seasonal"),
        ("Residual", "Residual Component", "Residual")
    ]

    fig, axes = plt.subplots(
        4, 1,
        figsize=(14, 9),
        sharex=True
    )

    for ax, (column, title, ylabel) in zip(
        axes, uni_plot_settings
    ):
        ax.plot(
            uni_zoom.index,
            uni_zoom[column],
            linewidth=0.8
        )
        ax.set_title(title, fontsize=11, loc="left")
        ax.set_ylabel(ylabel)
        ax.grid(True, linestyle="--", alpha=0.3)

    axes[-1].set_xlabel("Timestamp")

    fig.suptitle(
        f"{uni_target} - Time-Series Decomposition (12 Hours)",
        fontsize=14
    )

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    print("Decomposition period:", uni_period, "minutes")

else:
    print("The detected cycle is unsuitable for decomposition.")


In [ ]:
#Stationarity Check

def uni_check_stationarity(series, title):

    series = series.dropna()

    print(f"\n{title}")
    print("-" * 50)

    uni_start = (
        series.index[-1]
        - pd.Timedelta(hours=uni_plot_hours)
    )

    uni_zoom_series = series.loc[
        uni_start:series.index[-1]
    ]

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(
        uni_zoom_series.index,
        uni_zoom_series,
        linewidth=0.8
    )
    ax.set_title(f"{title} - Last 12 Hours", loc="left")
    ax.set_xlabel("Timestamp")
    ax.set_ylabel("Value")
    ax.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(12, 4))
    plot_acf(series, lags=30, ax=ax)
    ax.set_title(f"{title} - ACF (30 Lags)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    uni_adf = adfuller(series, autolag="AIC")

    print(f"ADF Statistic: {uni_adf[0]:.4f}")
    print(f"ADF p-value: {uni_adf[1]:.4f}")
    print(f"ADF supports stationarity: {uni_adf[1] <= 0.05}")

    uni_kpss = kpss(
        series,
        regression="c",
        nlags="auto"
    )

    print(f"KPSS Statistic: {uni_kpss[0]:.4f}")
    print(f"KPSS p-value: {uni_kpss[1]:.4f}")
    print(f"KPSS supports stationarity: {uni_kpss[1] > 0.05}")

    return uni_adf, uni_kpss


uni_adf_original, uni_kpss_original = uni_check_stationarity(
    uni_train,
    f"{uni_target} - Original Training Series"
)


In [ ]:
#Transformation

uni_diff = uni_train.diff().dropna()

uni_adf_diff, uni_kpss_diff = uni_check_stationarity(
    uni_diff,
    f"{uni_target} - First-Differenced Series"
)

uni_original_stationary = (
    uni_adf_original[1] <= 0.05
    and uni_kpss_original[1] > 0.05
)

uni_diff_stationary = (
    uni_adf_diff[1] <= 0.05
    and uni_kpss_diff[1] > 0.05
)

if uni_original_stationary:
    uni_d = 0

elif uni_diff_stationary:
    uni_d = 1

else:
    uni_d = 1

    print(
        "\nWarning: Stationarity tests are inconclusive. "
        "Using d=1 as a candidate; interpret results cautiously."
    )

print("\nSelected ARIMA differencing order d =", uni_d)


In [ ]:
#Normalization

uni_scaler = StandardScaler()

uni_train_scaled = pd.Series(
    uni_scaler.fit_transform(
        uni_train.to_numpy().reshape(-1, 1)
    ).ravel(),
    index=uni_train.index,
    name=uni_target
)

print("Original training mean:", round(uni_train.mean(), 4))
print("Original training standard deviation:", round(uni_train.std(), 4))
print("Scaled training mean:", round(uni_train_scaled.mean(), 4))
print("Scaled training standard deviation:", round(uni_train_scaled.std(), 4))

#Feature Understanding

if uni_d == 0:
    uni_structure_series = uni_train_scaled
else:
    uni_structure_series = uni_train_scaled.diff().dropna()

fig, axes = plt.subplots(
    2, 1,
    figsize=(12, 7)
)

plot_acf(
    uni_structure_series,
    lags=40,
    ax=axes[0]
)
axes[0].set_title("ACF - Stationarity-Adjusted LIT101")
axes[0].grid(True, alpha=0.3)

plot_pacf(
    uni_structure_series,
    lags=40,
    ax=axes[1]
)
axes[1].set_title("PACF - Stationarity-Adjusted LIT101")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("ACF helps suggest the moving-average order q.")
print("PACF helps suggest the autoregressive order p.")
print("Stationarity tests and differencing guide the order d.")


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

residuals = uni_candidate_fits[uni_best_order].resid

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

plot_acf(residuals, lags=60, ax=axes[0])
plot_pacf(residuals, lags=60, ax=axes[1])

axes[0].set_title("ARIMA Residual ACF")
axes[1].set_title("ARIMA Residual PACF")

plt.tight_layout()
plt.show()

In [ ]:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from statsmodels.stats.diagnostic import acorr_ljungbox

    model = SARIMAX(
        uni_train_scaled,
        order=(2, 0, 1),
        seasonal_order=(1, 1, 1, 69),
        trend="n",
        enforce_stationarity=True,
        enforce_invertibility=True
    )

    sarima_fit = model.fit(
        disp=True,
        maxiter=50
    )

    print("Converged:", sarima_fit.mle_retvals["converged"])
    print("AIC:", sarima_fit.aic)

    residuals = sarima_fit.resid.iloc[100:]

    display(
        acorr_ljungbox(
            residuals,
            lags=[20, 40, 80],
            model_df=4,
            return_df=True
        )
    )

In [ ]:
#Model Selection & Training

uni_trend = "c" if uni_d == 0 else "n"

uni_candidate_orders = [
    (4, 0, 2),
    (8, 0, 2),
    (10, 0, 1),
    (10, 0, 2),
    (12, 0, 1),
    (12, 0, 2),
    (14, 0, 1),
]
uni_candidate_fits = {}
uni_model_scores = []


def uni_calculate_metrics(actual, predicted):

    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    uni_mae_value = np.mean(
        np.abs(actual - predicted)
    )

    uni_mse_value = np.mean(
        (actual - predicted) ** 2
    )

    uni_rmse_value = np.sqrt(uni_mse_value)

    uni_nonzero = np.abs(actual) > 1e-8

    if uni_nonzero.any():
        uni_mape_value = np.mean(
            np.abs(
                (
                    actual[uni_nonzero]
                    - predicted[uni_nonzero]
                ) / actual[uni_nonzero]
            )
        ) * 100
    else:
        uni_mape_value = np.nan

    return {
        "MAE": uni_mae_value,
        "MSE": uni_mse_value,
        "RMSE": uni_rmse_value,
        "MAPE (%)": uni_mape_value
    }


for uni_order in uni_candidate_orders:

    uni_name = f"ARIMA{uni_order}"

    print(f"\nFitting {uni_name}...")

    try:
        uni_fit = ARIMA(
            uni_train_scaled,
            order=uni_order,
            trend=uni_trend
        ).fit()

        uni_val_forecast_scaled = np.asarray(
            uni_fit.forecast(
                steps=len(uni_validation)
            )
        )

        uni_val_forecast = uni_scaler.inverse_transform(
            uni_val_forecast_scaled.reshape(-1, 1)
        ).ravel()

        uni_val_actual = uni_validation.to_numpy()

        uni_val_metrics = uni_calculate_metrics(
            uni_val_actual,
            uni_val_forecast
        )

        uni_candidate_resid = np.asarray(uni_fit.resid)
        uni_model_df = uni_order[0] + uni_order[2]

        uni_candidate_lb = acorr_ljungbox(
            uni_candidate_resid,
            lags=[20],
            model_df=uni_model_df,
            return_df=True
        )

        uni_lb_pvalue = float(
            uni_candidate_lb["lb_pvalue"].iloc[0]
        )

        uni_candidate_fits[uni_order] = uni_fit

        uni_model_scores.append({
            "Model": uni_name,
            "Order": uni_order,
            "AIC": uni_fit.aic,
            "BIC": uni_fit.bic,
            "Validation MAE": uni_val_metrics["MAE"],
            "Validation MSE": uni_val_metrics["MSE"],
            "Validation RMSE": uni_val_metrics["RMSE"],
            "Validation MAPE (%)": uni_val_metrics["MAPE (%)"],
            "Ljung-Box p-value": uni_lb_pvalue,
            "Residuals pass test": uni_lb_pvalue > 0.05
        })

        print(f"AIC: {uni_fit.aic:.2f}")
        print(f"BIC: {uni_fit.bic:.2f}")
        print(f"Validation MAE: {uni_val_metrics['MAE']:.4f}")
        print(f"Validation MSE: {uni_val_metrics['MSE']:.4f}")
        print(f"Validation RMSE: {uni_val_metrics['RMSE']:.4f}")
        print(f"Validation MAPE: {uni_val_metrics['MAPE (%)']:.2f}%")
        print(f"Ljung-Box p-value: {uni_lb_pvalue:.6g}")

    except Exception as error:
        print(f"{uni_name} failed: {error}")


if not uni_model_scores:
    raise RuntimeError(
        "No ARIMA candidate fitted successfully. "
        "Check the data and stationarity decision."
    )

uni_comparison = pd.DataFrame(uni_model_scores)

display(
    uni_comparison
    .sort_values("Validation RMSE")
    .round(4)
)


uni_white_noise_candidates = [
    row for row in uni_model_scores
    if row["Residuals pass test"]
]

if uni_white_noise_candidates:
    uni_best_row = min(
        uni_white_noise_candidates,
        key=lambda row: row["Validation RMSE"]
    )
    print(
        "\nSelected from candidates passing the residual test, "
        "using lowest validation RMSE."
    )

else:
    uni_best_row = min(
        uni_model_scores,
        key=lambda row: row["Validation RMSE"]
    )

    print(
        "\nWarning: None of the tested ARIMA candidates passed "
        "the residual white-noise test. The lowest-validation-RMSE "
        "candidate is retained, but model limitations must be reported."
    )

uni_best_order = uni_best_row["Order"]
uni_best_name = uni_best_row["Model"]

print("\nSelected model:", uni_best_name)
print("Selected order:", uni_best_order)
print("Validation RMSE:", round(uni_best_row["Validation RMSE"], 4))


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))
plot_acf(uni_train_scaled, lags=240)
plt.title("LIT101 ACF: Checking for Seasonality")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

series = np.asarray(uni_train_scaled).ravel()
series = series - np.mean(series)

fft_values = np.fft.rfft(series)
frequencies = np.fft.rfftfreq(len(series), d=1)

power = np.abs(fft_values) ** 2

# Exclude zero frequency
valid = frequencies > 0

periods = 1 / frequencies[valid]
powers = power[valid]

# Investigate periods from 4 to 240 minutes
mask = (periods >= 4) & (periods <= 240)

top = pd.DataFrame({
    "Period (minutes)": periods[mask],
    "Power": powers[mask]
}).nlargest(10, "Power")

display(top)

plt.figure(figsize=(12, 4))
plt.plot(periods[mask], powers[mask])
plt.xlabel("Period (minutes)")
plt.ylabel("Spectral power")
plt.title("LIT101 FFT: Dominant Periods")
plt.grid(alpha=0.3)
plt.show()

print("Previously selected FFT period:", uni_period)

### Experimental SARIMA diagnostic, separate from final ARIMA pipeline

This cell is a *candidate test only*. The following Phase 9 to 11 cells still refit and evaluate **ordinary ARIMA**; do not claim the final pipeline uses SARIMA unless those cells are rewritten consistently. A Ljung-Box p-value above 0.05 at chosen lags is not a guarantee of white noise. Check approximately zero residual mean and stable variance too.

**Evaluation warning:** A single forecast covering the entire validation set is not comparable to the one-step-ahead evaluation used later in the notebook. Choose a forecast horizon and compare models on the same horizon. The 69-minute seasonal period is an FFT estimate, not proof that the seasonal cycle is fixed.

In [ ]:
# Phase 9 seasonal candidate: single SARIMA experiment (univariate LIT101)
# Uses the 69-minute period found on training data. Keep original ARIMA benchmark for comparison.
# simple_differencing=True reduces the seasonal state dimension by differencing up front.
# This does NOT guarantee a Ljung-Box pass; check convergence and diagnostics.
import time
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

seasonal_period = int(uni_period)
print('Seasonal period:', seasonal_period)
start = time.perf_counter()

sarima_model = SARIMAX(
    uni_train_scaled,
    order=(2, 0, 1),
    seasonal_order=(1, 1, 1, seasonal_period),
    trend='n',
    simple_differencing=True,
    enforce_stationarity=True,
    enforce_invertibility=True,
)

# Fit just ONE candidate. maxiter is not a time limit.
sarima_fit = sarima_model.fit(
    disp=True,
    maxiter=150
)

print("Converged:", sarima_fit.mle_retvals["converged"])
print("Iterations:", sarima_fit.mle_retvals.get("iterations"))
print("AIC:", sarima_fit.aic)

# Differencing shortens the series; first 100 remaining residuals excluded to
# reduce sensitivity to initial-state transients. Record this choice in report.
sarima_residuals = pd.Series(np.asarray(sarima_fit.resid)).iloc[100:]
print('Residual mean:', sarima_residuals.mean())
print('Residual variance:', sarima_residuals.var())
display(acorr_ljungbox(
    sarima_residuals, lags=[20, 40, 80], model_df=4, return_df=True
))


In [ ]:
# Phase 9: White Noise Check
residuals = sarima_fit.resid

plt.figure(figsize=(14, 10))

plt.subplot(3, 1, 1)
plt.plot(residuals)
plt.title("Residuals Over Time")
plt.axhline(y=0, color="r", linestyle="--", alpha=0.5)
plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 2)
plot_acf(residuals, lags=30, ax=plt.gca())
plt.title("Residual ACF - Should show no significant lags")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Forecast validation period
n = len(uni_validation)

forecast_scaled = np.asarray(
    sarima_fit.forecast(steps=n)
).reshape(-1, 1)

# Inverse scaling
forecast = uni_scaler.inverse_transform(
    forecast_scaled
).ravel()

actual = uni_validation.to_numpy().ravel()

# Evaluation
mae = mean_absolute_error(actual, forecast)
mse = mean_squared_error(actual, forecast)
rmse = np.sqrt(mse)
r2 = r2_score(actual, forecast)

print("SARIMA Validation Results")
print(f"MAE:  {mae:.4f}")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

print("\nPrevious ARIMA(4,0,2) RMSE: 110.3990")
print(f"SARIMA improvement: {110.3990 - rmse:.4f}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Scale validation data
val_scaled = pd.Series(
    uni_scaler.transform(
        uni_validation.to_numpy().reshape(-1, 1)
    ).ravel()
)

horizon = 60
results = []

models = {
    "ARIMA(4,0,2)": uni_candidate_fits[(4, 0, 2)],
    "SARIMA(2,0,1)(1,1,1,69)": sarima_fit
}

for name, fitted_model in models.items():

    print(f"Evaluating {name}...")

    predictions = []
    actuals = []

    updated_fit = fitted_model
    previous_origin = 0

    for origin in range(0, len(val_scaled) - horizon + 1, horizon):

        # Incorporate newly observed validation data
        if origin > previous_origin:
            new_data = val_scaled.iloc[previous_origin:origin]

            updated_fit = updated_fit.append(
                new_data.to_numpy(),
                refit=False
            )

        # Predict next 60 minutes
        forecast_scaled = np.asarray(
            updated_fit.forecast(steps=horizon)
        )

        forecast = uni_scaler.inverse_transform(
            forecast_scaled.reshape(-1, 1)
        ).ravel()

        actual = uni_validation.iloc[
            origin:origin + horizon
        ].to_numpy().ravel()

        predictions.extend(forecast)
        actuals.extend(actual)

        previous_origin = origin

    predictions = np.array(predictions)
    actuals = np.array(actuals)

    mae = mean_absolute_error(actuals, predictions)
    rmse = np.sqrt(mean_squared_error(actuals, predictions))

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse
    })

comparison = pd.DataFrame(results).sort_values("RMSE")

display(comparison)

In [ ]:
#Model Evaluation

uni_test_result = uni_best_model.get_forecast(
    steps=len(uni_test)
)

uni_test_forecast_values = uni_final_scaler.inverse_transform(
    np.asarray(
        uni_test_result.predicted_mean
    ).reshape(-1, 1)
).ravel()

uni_test_forecast = pd.Series(
    uni_test_forecast_values,
    index=uni_test.index,
    name="ARIMA Forecast"
)

uni_test_ci_scaled = uni_test_result.conf_int()

uni_test_lower = uni_final_scaler.inverse_transform(
    uni_test_ci_scaled.iloc[:, 0].to_numpy().reshape(-1, 1)
).ravel()

uni_test_upper = uni_final_scaler.inverse_transform(
    uni_test_ci_scaled.iloc[:, 1].to_numpy().reshape(-1, 1)
).ravel()

uni_test_ci = pd.DataFrame({
    "Lower 95% CI": uni_test_lower,
    "Upper 95% CI": uni_test_upper
}, index=uni_test.index)

uni_test_metrics_values = uni_calculate_metrics(
    uni_test.to_numpy(),
    uni_test_forecast.to_numpy()
)

uni_test_metrics = pd.DataFrame(
    [{
        **uni_test_metrics_values,
        "AIC": uni_best_model.aic,
        "BIC": uni_best_model.bic
    }],
    index=[uni_best_name]
)

print("Final test-set evaluation:")
display(uni_test_metrics.round(4))


In [ ]:
# target plus the flows into and out of the same tank
var_columns = ["LIT101", "FIT101", "FIT201"]
var_data = normal_1min[var_columns].asfreq("1min")
var_train = var_data[var_data.index < TEST_START]
var_test_index = var_data.index[var_data.index >= TEST_START]

# stationarity of each series on the training period, both tests
var_rows = []
for col in var_columns:
    x = var_train[col]
    adf_p = adfuller(x, autolag="AIC")[1]
    kpss_p = kpss(x, regression="c", nlags="auto")[1]
    var_rows.append({"Variable": col, "ADF p-value": adf_p, "KPSS p-value": kpss_p,
                     "Stationary": adf_p <= 0.05 and kpss_p > 0.05})
var_stationarity = pd.DataFrame(var_rows)
display(var_stationarity.round(4))

# difference all series only if any of them is non-stationary
var_d = 0 if var_stationarity["Stationary"].all() else 1
var_work = var_data.diff().dropna() if var_d else var_data
print("Differencing order:", var_d)
print(f"Train: {var_train.index[0]} to {var_train.index[-1]} | Test: {var_test_index[0]} to {var_test_index[-1]}")

In [ ]:
# lag order chosen by AIC on the training period
var_fit_data = var_work[var_work.index < TEST_START]
var_lag = VAR(var_fit_data).select_order(maxlags=CYCLE).aic
var_model = VAR(var_fit_data).fit(var_lag)
print("Lag selected by AIC:", var_lag, "| stable:", var_model.is_stable())

# one-step forecasts: each test minute predicted from the true previous var_lag minutes
var_pred = []
for t in var_test_index:
    pos = var_work.index.get_loc(t)
    history = var_work.iloc[pos - var_lag:pos].to_numpy()
    step = var_model.forecast(history, steps=1)[0, 0]   # column 0 = LIT101
    # with differencing, the forecast is a change, so add it to the previous true level
    var_pred.append(step + var_data["LIT101"].shift(1).loc[t] if var_d else step)
var_pred = pd.Series(var_pred, index=var_test_index)
var_actual = var_data.loc[var_test_index, "LIT101"]

In [ ]:
# test metrics
display(pd.DataFrame([regression_metrics(var_actual, var_pred)], index=["VAR (one-step)"]).round(4))

# actual vs predicted, first 4 hours of the test period
plt.figure(figsize=(14, 5))
plt.plot(var_test_index[:240], var_actual.iloc[:240], label="Actual")
plt.plot(var_test_index[:240], var_pred.iloc[:240], "--", label="VAR one-step forecast")
plt.title("VAR: actual vs predicted LIT101 (first 4 hours of test)")
plt.xlabel("Time"); plt.ylabel("Water level (mm)")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# residuals of the LIT101 equation on the training period: should look like white noise
var_resid = var_model.resid["LIT101"]
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].plot(var_resid.iloc[-uni_plot_hours * 60:], linewidth=0.7)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("VAR residuals (LIT101 equation, last 12 hours of training)")
plot_acf(var_resid, lags=2 * CYCLE + 5, ax=axes[1]); axes[1].set_title("VAR residual ACF")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

display(acorr_ljungbox(var_resid, lags=[CYCLE, 2 * CYCLE], return_df=True))

In [ ]:
# every model's test predictions, keyed by timestamp
all_predictions = {
    "Persistence (previous minute)": pd.Series(X_test_full["lag_1"].values, index=X_test_full.index),
    f"{uni_best_name} (multi-step)": uni_test_forecast,
    f"{uni_best_name} (one-step)": uni_onestep,
    "VAR (one-step)": var_pred,
    "CatBoost": pd.Series(y_pred_cb, index=y_test_final.index),
    "XGBoost (tuned)": pd.Series(y_pred_xgb, index=y_test_final.index),
}

# score every model on the minutes they all cover, so the comparison is fair
actual_all = ml_df[target]
common = y_test_final.index
for p in all_predictions.values():
    common = common.intersection(p.index)

final_comparison = pd.DataFrame({name: regression_metrics(actual_all.loc[common], p.loc[common])
                                 for name, p in all_predictions.items()}).T.sort_values("RMSE")
print(f"Scored on {len(common)} common test minutes: {common.min()} to {common.max()}")
display(final_comparison.round(4))

final_comparison[["MAE", "RMSE"]].plot.barh(figsize=(10, 5))
plt.title("Test error by model on the same minutes"); plt.xlabel("Error (mm)")
plt.grid(True, axis="x", alpha=0.3); plt.tight_layout(); plt.show()